<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session06-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 6: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Myoglobin and haemoglobin: does quaternary structure change secondary structure?

Myoglobin (PDB `1MBN`, solved by Kendrew in 1958, the first protein structure ever determined) is
haemoglobin's evolutionary cousin: in Session 4's globin tree, myoglobin branches off before the
haemoglobin chains split into α and β. Myoglobin stores oxygen in muscle as a **single chain**: it has
no quaternary structure, unlike haemoglobin's four-chain assembly.

PyMOL's `dss` algorithm, run on the atomic coordinates, finds haemoglobin's β chain 80.8% helix and
0% strand. What does it find for myoglobin, which works alone?

**Fix and run.** The first cell installs the libraries Colab lacks. The second downloads myoglobin,
assigns its secondary structure and counts the residues in helices and strands. One character in the
line marked `FIX` is wrong. Run it once as it is and look at the result; then fix the line and run it
again. (PyMOL labels each residue `H` for helix, `S` for strand and `L` for loop.)

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython"), ("pymol", "pymol-open-source-whl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
from Bio.PDB import PDBList
from pymol import cmd                  # PyMOL as a Python library, no window needed

mb_path = PDBList(verbose=False).retrieve_pdb_file("1mbn", pdir=".", file_format="pdb")
cmd.load(mb_path, "mb")
cmd.dss("mb")                          # assign helix, strand and loop from the coordinates

ss_list = []                           # one label per residue: its C-alpha atom
cmd.iterate("mb and polymer and name CA", "ss_list.append(ss)", space={"ss_list": ss_list})

total = len(ss_list)
helix = sum(1 for s in ss_list if s == "h")     # FIX: one character in this line is wrong
strand = sum(1 for s in ss_list if s == "S")
print(f"{total} residues: {100 * helix / total:.1f}% helix, {100 * strand / total:.1f}% strand")

153 residues: 0.0% helix, 0.0% strand


**If you finish early.**

1. Print `"".join(ss_list)`. Where are the loops: at the ends, or between the helices?
2. If the fold does not need four chains, why does haemoglobin have them? Think about what it has to
   do in the lungs and in the tissues that myoglobin does not.

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the count compared each label with a lowercase `"h"`, while
PyMOL writes `"H"`. Nothing matched, so the cell reported 0% helix and 0% strand, as if myoglobin
were all loop. Fixed (`s == "H"`), myoglobin comes out at **79.7% helix, 0% strand** (122 of 153
residues): essentially the same as haemoglobin's β chain (80.8%, 0%). A result of exactly zero is
worth a second look before you believe it.

Secondary and tertiary structure are properties of *one folded chain*; they do not depend on
whether that chain then assembles with others. Quaternary structure is a separate level on top. So
why does haemoglobin need four chains? **Cooperativity**: haemoglobin must bind oxygen in the lungs
and *release* it in the tissues, and its four subunits signal each other (the T to R shift of block
3), so that binding at one subunit makes the others bind more easily. Myoglobin only stores oxygen
at one fixed affinity in muscle, and one chain is enough for that.

</details>